In [ ]:
# ============================================================
# CELDA 1 — Instalar dependencias
# ============================================================
!pip install ultralytics -q

import os, shutil, zipfile, cv2, random
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# ============================================================
# CELDA 2 — Descargar tu dataset + los 3 externos
# ============================================================
!kaggle datasets download -d davidgomez32/coles-plagas-enfermedades-agrobot \
    -p /kaggle/working/mi_dataset --unzip -q

!kaggle datasets download -d madhukk23/balanced-cauliflower-dataset-200-each \
    -p /kaggle/working/ext1 --unzip -q

!kaggle datasets download -d shuvokumarbasak2030/cauliflower-disease-multi-transformation-dataset \
    -p /kaggle/working/ext2 --unzip -q

!kaggle datasets download -d drtawfikrrahman/deep-learning-model-for-potato-pest-detection \
    -p /kaggle/working/ext3 --unzip -q

print("✅ Todos los datasets descargados")

# ============================================================
# CELDA 3 — Descomprimir tu dataset
# ============================================================
BASE_MIO = "/kaggle/working/mi_dataset"
for z in ["images.zip", "labels.zip"]:
    ruta = os.path.join(BASE_MIO, z)
    if os.path.exists(ruta):
        with zipfile.ZipFile(ruta, 'r') as f:
            f.extractall(BASE_MIO)
        print(f"✅ {z} descomprimido")

# ============================================================
# CELDA 4 — Ver estructura de datasets externos
# ============================================================
for nombre, ruta in [("EXT1", "/kaggle/working/ext1"),
                     ("EXT2", "/kaggle/working/ext2"),
                     ("EXT3", "/kaggle/working/ext3")]:
    print(f"\n📂 {nombre}:")
    for r, dirs, files in os.walk(ruta):
        level = r.replace(ruta, "").count(os.sep)
        if level < 3:
            print("  " * level + os.path.basename(r) + f"/  ({len(files)} archivos)")

# ============================================================
# CELDA 5 — Mapeo: categorías externas → tus clases YOLO
# ============================================================
# Tus clases:
# 0: col_sana        1: col_amarillamiento   2: col_manchas_cafe
# 3: col_dano_insecto  4: col_mildiu_velloso   5: col_malformacion
# 6: col_pudricion

MAPA_CLASES = {
    # Dataset 1 (balanced cauliflower)
    "Healthy":                  0,   # col_sana
    "No disease":               0,   # col_sana
    "Black Rot":                2,   # col_manchas_cafe
    "Downy Mildew":             4,   # col_mildiu_velloso
    "Soft Rot":                 6,   # col_pudricion
    "Bacterial Spot":           2,   # col_manchas_cafe

    # Dataset 2 (cauliflower multi-transformation)
    "healthy":                  0,
    "black_rot":                2,
    "downy_mildew":             4,
    "soft_rot":                 6,
    "Blackrot":                 2,
    "DownyMildew":              4,

    # Dataset 3 (potato pest)
    "Cabbage aphid colony":     3,   # col_dano_insecto
    "Green Peach Aphid":        3,   # col_dano_insecto
    "Potato Leafminer":         3,   # col_dano_insecto
    "Leaf Miner":               3,   # col_dano_insecto
    "Aphid":                    3,   # col_dano_insecto
}

# ============================================================
# CELDA 6 — Función de calidad de imagen
# ============================================================
def es_imagen_apta(ruta, umbral_nitidez=80.0, umbral_sujeto=0.03):
    img = cv2.imread(ruta)
    if img is None:
        return False
    gris = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    nitidez = cv2.Laplacian(gris, cv2.CV_64F).var()
    _, mask = cv2.threshold(gris, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    proporcion = np.count_nonzero(mask) / mask.size
    return nitidez >= umbral_nitidez and proporcion >= umbral_sujeto

# ============================================================
# CELDA 7 — Recopilar imágenes externas aptas
# ============================================================
EXTERNAS = []  # Lista de (ruta_imagen, clase_id)
DIRECTORIOS_EXT = [
    "/kaggle/working/ext1",
    "/kaggle/working/ext2",
    "/kaggle/working/ext3",
]

for base in DIRECTORIOS_EXT:
    for raiz, dirs, archivos in os.walk(base):
        nombre_cat = os.path.basename(raiz)
        if nombre_cat not in MAPA_CLASES:
            continue
        clase_id = MAPA_CLASES[nombre_cat]
        for archivo in archivos:
            if not archivo.lower().endswith(('.jpg', '.jpeg', '.png')):
                continue
            ruta = os.path.join(raiz, archivo)
            if es_imagen_apta(ruta):
                EXTERNAS.append((ruta, clase_id))

print(f"✅ Imágenes externas aptas encontradas: {len(EXTERNAS)}")

# Distribución por clase
from collections import Counter
dist = Counter(c for _, c in EXTERNAS)
nombres = ["col_sana","col_amarillamiento","col_manchas_cafe",
           "col_dano_insecto","col_mildiu_velloso","col_malformacion","col_pudricion"]
for cid, cnt in sorted(dist.items()):
    print(f"  Clase {cid} ({nombres[cid]}): {cnt} imágenes")

# ============================================================
# CELDA 8 — Balanceo: máx 200 por clase (evitar desbalance)
# ============================================================
MAX_POR_CLASE = 200
random.seed(42)

from collections import defaultdict
por_clase = defaultdict(list)
for ruta, cid in EXTERNAS:
    por_clase[cid].append(ruta)

EXTERNAS_BALANCEADAS = []
for cid, rutas in por_clase.items():
    seleccionadas = random.sample(rutas, min(len(rutas), MAX_POR_CLASE))
    EXTERNAS_BALANCEADAS.extend([(r, cid) for r in seleccionadas])

random.shuffle(EXTERNAS_BALANCEADAS)
print(f"✅ Imágenes balanceadas: {len(EXTERNAS_BALANCEADAS)}")

# ============================================================
# CELDA 9 — Dataset de salida combinado
# ============================================================
SALIDA = "/kaggle/working/dataset_combinado"
for split in ["train", "val", "test"]:
    os.makedirs(f"{SALIDA}/images/{split}", exist_ok=True)
    os.makedirs(f"{SALIDA}/labels/{split}", exist_ok=True)

# --- Copiar tu dataset original ---
for split in ["train", "val", "test"]:
    src_img = f"{BASE_MIO}/images/{split}"
    src_lbl = f"{BASE_MIO}/labels/{split}"
    if not os.path.exists(src_img):
        continue
    for img_file in os.listdir(src_img):
        shutil.copy2(f"{src_img}/{img_file}", f"{SALIDA}/images/{split}/{img_file}")
        lbl_file = img_file.rsplit(".", 1)[0] + ".txt"
        lbl_src = f"{src_lbl}/{lbl_file}"
        if os.path.exists(lbl_src):
            shutil.copy2(lbl_src, f"{SALIDA}/labels/{split}/{lbl_file}")

print("✅ Dataset original copiado")

# --- Agregar imágenes externas (70/20/10) ---
n = len(EXTERNAS_BALANCEADAS)
cortes = [int(n * 0.70), int(n * 0.90)]
splits_ext = (
    ["train"] * cortes[0] +
    ["val"]   * (cortes[1] - cortes[0]) +
    ["test"]  * (n - cortes[1])
)

for i, (ruta, cid) in enumerate(EXTERNAS_BALANCEADAS):
    split = splits_ext[i]
    ext = Path(ruta).suffix

    # Redimensionar a 640x640
    img = cv2.imread(ruta)
    img_res = cv2.resize(img, (640, 640))

    nombre_base = f"ext_{i:05d}"
    cv2.imwrite(f"{SALIDA}/images/{split}/{nombre_base}.jpg", img_res)

    # Bbox = imagen completa (cx cy w h normalizados)
    # Para imágenes de clasificación sin bbox, usamos bbox completo
    with open(f"{SALIDA}/labels/{split}/{nombre_base}.txt", "w") as f:
        f.write(f"{cid} 0.5 0.5 1.0 1.0\n")

print(f"✅ {len(EXTERNAS_BALANCEADAS)} imágenes externas agregadas")

# ============================================================
# CELDA 10 — dataset.yaml
# ============================================================
yaml = f"""path: {SALIDA}
train: images/train
val:   images/val
test:  images/test

nc: 7
names:
  0: col_sana
  1: col_amarillamiento
  2: col_manchas_cafe
  3: col_dano_insecto
  4: col_mildiu_velloso
  5: col_malformacion
  6: col_pudricion
"""
with open(f"{SALIDA}/dataset.yaml", "w") as f:
    f.write(yaml)

# Conteo final
for split in ["train", "val", "test"]:
    n_imgs = len(os.listdir(f"{SALIDA}/images/{split}"))
    print(f"  {split}: {n_imgs} imágenes")

# ============================================================
# CELDA 11 — Entrenar YOLOv8 con dataset combinado
# ============================================================
from ultralytics import YOLO

model = YOLO("yolov8s.pt")   # 's' = small, mejor que 'n' con más datos

results = model.train(
    data     = f"{SALIDA}/dataset.yaml",
    epochs   = 150,
    imgsz    = 640,
    batch    = 16,
    name     = "coles_combinado_v2",
    project  = "/kaggle/working/runs",
    patience = 25,
    device   = 0,
    # Augmentación
    hsv_h    = 0.015,
    hsv_s    = 0.7,
    hsv_v    = 0.4,
    flipud   = 0.3,
    fliplr   = 0.5,
    mosaic   = 1.0,
    mixup    = 0.1,
)

# ============================================================
# CELDA 12 — Métricas y exportar ONNX
# ============================================================
metrics = model.val()
print(f"\nmAP50:    {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")

best = "/kaggle/working/runs/coles_combinado_v2/weights/best.pt"
YOLO(best).export(format="onnx", imgsz=640, simplify=True)
print("✅ best.onnx listo para Jetson Orin Nano")